# Modelling (Multivariate Analyses) – Portfolio opdrachten 15 t/m 18

In deze notebook werk ik de **portfolio-opdrachten 15 t/m 18** uit uit *Multivariate Analyses.ipynb*.

Datasets:
- **Penguins**: `penguins.csv` (automatisch te downloaden)  
- **Minecraft speedrun dataset**: `mce.csv` (meegeleverd)

> Tip (Deepnote): zorg dat `mce.csv` in dezelfde projectmap staat als deze notebook.


## Setup

We gebruiken o.a. `pandas`, `scikit-learn` en `graphviz` om Decision Trees te trainen en te visualiseren.


In [ ]:
# (Optioneel) Installeer dependencies als je omgeving ze nog niet heeft
!pip -q install pandas numpy scikit-learn matplotlib graphviz

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report

import graphviz
from sklearn import tree


ModuleNotFoundError: No module named 'graphviz'

## Helperfuncties

Deze helperfuncties komen overeen met de functies uit *Multivariate Analyses.ipynb* (accuracy, RMSE en tree-plots).


In [ ]:
def calculate_accuracy(predictions, actuals):
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    if len(predictions) != len(actuals):
        raise ValueError("The amount of predictions did not equal the amount of actuals")
    return (predictions == actuals).sum() / len(actuals)

def calculate_rmse(predictions, actuals):
    predictions = np.array(predictions, dtype=float)
    actuals = np.array(actuals, dtype=float)
    if len(predictions) != len(actuals):
        raise ValueError("The amount of predictions did not equal the amount of actuals")
    return np.sqrt(np.mean((predictions - actuals) ** 2))

def plot_tree_classification(model, feature_names, class_names):
    dot_data = tree.export_graphviz(
        model,
        out_file=None,
        feature_names=feature_names,
        class_names=[str(c) for c in class_names],
        filled=True,
        rounded=True,
        special_characters=True
    )
    graph = graphviz.Source(dot_data)
    return graph

def plot_tree_regression(model, feature_names):
    dot_data = tree.export_graphviz(
        model,
        out_file=None,
        feature_names=feature_names,
        filled=True,
        rounded=True,
        special_characters=True
    )
    graph = graphviz.Source(dot_data)
    return graph


## Data laden

### Penguins
We proberen eerst `penguins.csv` lokaal te laden. Als die er niet is, downloaden we de CSV uit de officiële `seaborn-data` repository.

### Minecraft speedrun data
We laden `mce.csv` vanuit de projectmap.


In [ ]:
# --- Penguins dataset ---
PENGUINS_PATH = "penguins.csv"
if not os.path.exists(PENGUINS_PATH):
    # download (werkt in omgevingen met internet; Deepnote meestal wel)
    import urllib.request
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
    urllib.request.urlretrieve(url, PENGUINS_PATH)

penguins = pd.read_csv(PENGUINS_PATH)
display(penguins.head())
print("Penguins shape:", penguins.shape)

# --- Minecraft speedrun dataset ---
MCE_PATH = "mce.csv"
if not os.path.exists(MCE_PATH) and os.path.exists("/mnt/data/mce.csv"):
    MCE_PATH = "/mnt/data/mce.csv"

mce = pd.read_csv(MCE_PATH)
display(mce.head())
print("MCE shape:", mce.shape)


## Portfolio opdracht 15 – DecisionTreeClassifier (penguins)

**Doel:** voorspel de **species** van een penguin op basis van kenmerken.

Stappen:
1. Data voorbereiden (missing values oplossen).
2. Train/test split (70/30).
3. DecisionTreeClassifier trainen.
4. Voorspellen op train én test.
5. Evalueren (accuracy + confusion matrix) en een boom visualiseren.


In [ ]:
# --- Features/target ---
X = penguins.drop(columns=["species"])
y = penguins["species"]

numeric_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
categorical_features = ["island", "sex"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=4, random_state=42)

clf = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])

clf.fit(X_train, y_train)

pred_train = clf.predict(X_train)
pred_test = clf.predict(X_test)

train_acc = calculate_accuracy(pred_train, y_train)
test_acc = calculate_accuracy(pred_test, y_test)

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test  accuracy: {test_acc:.4f}")

# Extra evaluatie (handig bij multi-class)
print("\nClassification report (test):")
print(classification_report(y_test, pred_test))

print("Confusion matrix (test):")
labels = clf.named_steps["model"].classes_
cm = confusion_matrix(y_test, pred_test, labels=labels)
display(pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels]))


### (Kort) experiment: depth & feature-set

De opdracht vraagt niet expliciet om meerdere iteraties, maar het helpt om te zien wat `max_depth` en feature-keuze doen.


In [ ]:
def eval_penguin_classifier(feature_cols, max_depth):
    X = penguins[feature_cols]
    y = penguins["species"]

    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(penguins[c])]
    cat_cols = [c for c in feature_cols if c not in num_cols]

    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    pipe = Pipeline([
        ("preprocess", pre),
        ("model", DecisionTreeClassifier(max_depth=max_depth, random_state=42))
    ])
    pipe.fit(X_train, y_train)

    return {
        "features": ", ".join(feature_cols),
        "max_depth": max_depth,
        "train_acc": accuracy_score(y_train, pipe.predict(X_train)),
        "test_acc": accuracy_score(y_test, pipe.predict(X_test)),
    }

experiments = [
    eval_penguin_classifier(["bill_length_mm", "bill_depth_mm"], 2),
    eval_penguin_classifier(["bill_length_mm", "bill_depth_mm", "flipper_length_mm"], 3),
    eval_penguin_classifier(["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "island", "sex"], 4),
]

display(pd.DataFrame(experiments))


### Visualisatie van de decision tree (opdracht 15)

We visualiseren het model dat hierboven is getraind. Omdat we one-hot encoding gebruiken, krijgen we **geëxpandeerde feature-namen**.


In [ ]:
# Feature namen na preprocessing (one-hot)
feature_names = clf.named_steps["preprocess"].get_feature_names_out()
feature_names = [n.replace("num__", "").replace("cat__", "") for n in feature_names]

plot_tree_classification(
    clf.named_steps["model"],
    feature_names=feature_names,
    class_names=clf.named_steps["model"].classes_,
)


**Findings (opdracht 15)**  
- In mijn run is de train-accuracy hoger dan de test-accuracy (lichte overfitting is normaal bij bomen).  
- De confusion matrix laat zien welke species het model soms verwart (meestal zijn dat de classes die qua kenmerken dichter bij elkaar liggen).  


## Portfolio opdracht 16 – DecisionTreeClassifier (eigen dataset: `mce.csv`)

**Doel:** voorspel een **categorische** kolom uit je eigen dataset.

Ik kies hier als target `cat_name` (de categorie-naam van de speedrun).  
Omdat er een paar categorieën extreem weinig voorkomen (bijv. 1 record), maak ik een extra kolom `cat_name_grouped` waarbij zeldzame categorieën worden samengevoegd naar **Other**. Dat maakt de train/test-split (met stratify) stabieler.


In [ ]:
# --- Feature engineering (datums) ---
mce_fe = mce.copy()

for col in ["verify_date", "submitted_date", "player_signup_date"]:
    mce_fe[col] = pd.to_datetime(mce_fe[col], errors="coerce")

mce_fe["submitted_year"] = mce_fe["submitted_date"].dt.year
mce_fe["submitted_month"] = mce_fe["submitted_date"].dt.month
mce_fe["submitted_dow"] = mce_fe["submitted_date"].dt.dayofweek
mce_fe["days_since_signup"] = (mce_fe["submitted_date"] - mce_fe["player_signup_date"]).dt.days

# --- Target: cat_name, maar zeldzame classes -> Other ---
counts = mce_fe["cat_name"].value_counts()
rare_classes = counts[counts < 5].index
mce_fe["cat_name_grouped"] = mce_fe["cat_name"].where(~mce_fe["cat_name"].isin(rare_classes), other="Other")

display(mce_fe["cat_name_grouped"].value_counts())

# --- Features kiezen ---
feature_cols = [
    "speedrun_time",
    "place",
    "platform_released_year",
    "is_level_cat",
    "platform_name",
    "player_country",
    "submitted_year",
    "submitted_month",
    "submitted_dow",
    "days_since_signup",
]

X = mce_fe[feature_cols]
y = mce_fe["cat_name_grouped"]

num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(mce_fe[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

dtc = DecisionTreeClassifier(max_depth=5, random_state=42)

clf_mce = Pipeline([
    ("preprocess", preprocess),
    ("model", dtc),
])

clf_mce.fit(X_train, y_train)

pred_train = clf_mce.predict(X_train)
pred_test = clf_mce.predict(X_test)

train_acc = calculate_accuracy(pred_train, y_train)
test_acc = calculate_accuracy(pred_test, y_test)

# Bij sterke class-imbalance is 'balanced accuracy' vaak eerlijker
bal_test_acc = balanced_accuracy_score(y_test, pred_test)

print(f"Train accuracy:         {train_acc:.4f}")
print(f"Test accuracy:          {test_acc:.4f}")
print(f"Balanced test accuracy: {bal_test_acc:.4f}")

print("\nClassification report (test):")
print(classification_report(y_test, pred_test, zero_division=0))


### Iteraties (opdracht 16): welke depth en features per cycle?

Hieronder 3 simpele cycles:
1. Alleen 2 numerieke features  
2. Numeriek + een paar categorische features  
3. De volledige set + iets diepere boom


In [ ]:
def eval_mce_classifier(feature_cols, max_depth):
    X = mce_fe[feature_cols]
    y = mce_fe["cat_name_grouped"]

    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(mce_fe[c])]
    cat_cols = [c for c in feature_cols if c not in num_cols]

    pre = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    pipe = Pipeline([
        ("preprocess", pre),
        ("model", DecisionTreeClassifier(max_depth=max_depth, random_state=42))
    ])
    pipe.fit(X_train, y_train)

    yhat_test = pipe.predict(X_test)

    return {
        "features": ", ".join(feature_cols),
        "max_depth": max_depth,
        "train_acc": accuracy_score(y_train, pipe.predict(X_train)),
        "test_acc": accuracy_score(y_test, yhat_test),
        "balanced_test_acc": balanced_accuracy_score(y_test, yhat_test),
    }

cycles = [
    eval_mce_classifier(["speedrun_time", "platform_released_year"], 3),
    eval_mce_classifier(["speedrun_time", "platform_released_year", "platform_name", "is_level_cat", "player_country"], 3),
    eval_mce_classifier(feature_cols, 5),
]

display(pd.DataFrame(cycles))


### Visualisatie decision tree (opdracht 16)

Omdat er one-hot encoding is gebruikt, plotten we de boom met de gegenereerde feature-namen.
(Praktisch: hou `max_depth` niet te hoog, anders wordt de plot enorm.)


In [ ]:
feature_names = clf_mce.named_steps["preprocess"].get_feature_names_out()
feature_names = [n.replace("num__", "").replace("cat__", "") for n in feature_names]

plot_tree_classification(
    clf_mce.named_steps["model"],
    feature_names=feature_names,
    class_names=clf_mce.named_steps["model"].classes_,
)


**Findings (opdracht 16)**  
- De **gewone accuracy** kan hoog lijken omdat één categorie (Time Attack) extreem vaak voorkomt.  
- Daarom is **balanced accuracy** een nuttige extra metric om te kijken of het model ook de kleinere classes ‘ziet’.  
- Als je het nog beter wilt maken: probeer `class_weight="balanced"` of verzamel/merge classes op een andere manier.  


## Portfolio opdracht 17 – DecisionTreeRegressor (penguins)

**Doel:** voorspel `body_mass_g` op basis van penguin-kenmerken.

Stappen:
1. Missing target-waarden verwijderen (anders kan het model niet trainen).
2. Train/test split (70/30).
3. DecisionTreeRegressor trainen.
4. RMSE op train én test berekenen.
5. Tree visualiseren en interpreteren.


In [ ]:
peng_reg = penguins.dropna(subset=["body_mass_g"]).copy()

target = "body_mass_g"
feature_cols = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm", "species", "sex", "island"]

X = peng_reg[feature_cols]
y = peng_reg[target]

num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(peng_reg[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

dtr = DecisionTreeRegressor(max_depth=2, random_state=42)

reg_peng = Pipeline([
    ("preprocess", preprocess),
    ("model", dtr),
])
reg_peng.fit(X_train, y_train)

pred_train = reg_peng.predict(X_train)
pred_test = reg_peng.predict(X_test)

rmse_train = calculate_rmse(pred_train, y_train)
rmse_test = calculate_rmse(pred_test, y_test)

print(f"RMSE train: {rmse_train:.2f}")
print(f"RMSE test:  {rmse_test:.2f}")


### Iteraties (opdracht 17): depth & features

Een simpele manier om iteratief te werken is 2–3 cycles uit te proberen.  
Let op: **als de test-RMSE slechter wordt terwijl train-RMSE beter wordt, is dat vaak overfitting.**


In [ ]:
def eval_penguin_regressor(feature_cols, max_depth):
    data = penguins.dropna(subset=["body_mass_g"]).copy()
    X = data[feature_cols]
    y = data["body_mass_g"]

    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(data[c])]
    cat_cols = [c for c in feature_cols if c not in num_cols]

    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

    pipe = Pipeline([
        ("preprocess", pre),
        ("model", DecisionTreeRegressor(max_depth=max_depth, random_state=42))
    ])
    pipe.fit(X_train, y_train)

    return {
        "features": ", ".join(feature_cols),
        "max_depth": max_depth,
        "rmse_train": calculate_rmse(pipe.predict(X_train), y_train),
        "rmse_test": calculate_rmse(pipe.predict(X_test), y_test),
    }

cycles = [
    eval_penguin_regressor(["flipper_length_mm"], 2),
    eval_penguin_regressor(["flipper_length_mm", "bill_length_mm", "bill_depth_mm"], 3),
    eval_penguin_regressor(["flipper_length_mm", "bill_length_mm", "bill_depth_mm", "species", "sex", "island"], 2),
]
display(pd.DataFrame(cycles))


### Visualisatie decision tree (opdracht 17)

We plotten de getrainde boom (depth 2 is meestal goed te lezen).


In [ ]:
feature_names = reg_peng.named_steps["preprocess"].get_feature_names_out()
feature_names = [n.replace("num__", "").replace("cat__", "") for n in feature_names]

plot_tree_regression(
    reg_peng.named_steps["model"],
    feature_names=feature_names,
)


**Findings (opdracht 17)**  
- Meestal geldt: RMSE op **train** < RMSE op **test** (trainset is ‘bekender’).  
- Als het verschil groot wordt bij hogere `max_depth`, is dat een signaal van overfitting.  
- Met een kleine `max_depth` kun je de boom beter interpreteren: welke splits zijn het belangrijkst?  


## Portfolio opdracht 18 – DecisionTreeRegressor (eigen dataset: `mce.csv`)

**Doel:** voorspel een **numerieke** kolom uit je eigen dataset.

Ik kies als target `speedrun_time` (tijd in seconden).  
In deze dataset zitten een paar **extreme uitschieters** (bijv. 40.320 seconden). Daardoor wordt RMSE enorm en minder informatief.

Daarom laat ik twee varianten zien:
1. **Volledige dataset** (met outliers)
2. **Gefilterd** op de 95e percentiel (top 5% snelheden eruit) – dit geeft meestal een beter interpreteerbare RMSE.


In [ ]:
mce_reg = mce.copy()

for col in ["verify_date", "submitted_date", "player_signup_date"]:
    mce_reg[col] = pd.to_datetime(mce_reg[col], errors="coerce")

mce_reg["submitted_year"] = mce_reg["submitted_date"].dt.year
mce_reg["submitted_month"] = mce_reg["submitted_date"].dt.month
mce_reg["submitted_dow"] = mce_reg["submitted_date"].dt.dayofweek
mce_reg["days_since_signup"] = (mce_reg["submitted_date"] - mce_reg["player_signup_date"]).dt.days

target = "speedrun_time"

feature_cols = [
    "place",
    "platform_released_year",
    "is_level_cat",
    "platform_name",
    "player_country",
    "cat_name",
    "submitted_year",
    "submitted_month",
    "submitted_dow",
    "days_since_signup",
]

def train_eval_speedrun_regressor(data, max_depth=6):
    X = data[feature_cols]
    y = data[target]

    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(data[c])]
    cat_cols = [c for c in feature_cols if c not in num_cols]

    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

    pipe = Pipeline([
        ("preprocess", pre),
        ("model", DecisionTreeRegressor(max_depth=max_depth, random_state=42))
    ])
    pipe.fit(X_train, y_train)

    rmse_train = calculate_rmse(pipe.predict(X_train), y_train)
    rmse_test = calculate_rmse(pipe.predict(X_test), y_test)
    baseline_rmse = calculate_rmse(np.repeat(y_train.mean(), len(y_test)), y_test)

    return pipe, {
        "rows": len(data),
        "max_depth": max_depth,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "baseline_rmse_test": baseline_rmse,
    }

# Variant 1: full dataset
pipe_full, res_full = train_eval_speedrun_regressor(mce_reg, max_depth=6)

# Variant 2: filter outliers (<= 95e percentiel)
p95 = mce_reg[target].quantile(0.95)
mce_filtered = mce_reg[mce_reg[target] <= p95].copy()

pipe_filt, res_filt = train_eval_speedrun_regressor(mce_filtered, max_depth=6)

display(pd.DataFrame([res_full, res_filt]))
print("95e percentiel speedrun_time =", p95)


### Iteraties (opdracht 18): depth & features per cycle

Onderstaand 3 cycles op de **gefilterde** dataset:
1. Baseline met 2 numerieke features  
2. + `cat_name` (categorie is vaak sterk voorspellend voor tijd)  
3. Volledige featureset + diepere boom


In [ ]:
def eval_mce_regressor(data, feature_cols, max_depth):
    X = data[feature_cols]
    y = data["speedrun_time"]

    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(data[c])]
    cat_cols = [c for c in feature_cols if c not in num_cols]

    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

    pipe = Pipeline([
        ("preprocess", pre),
        ("model", DecisionTreeRegressor(max_depth=max_depth, random_state=42))
    ])
    pipe.fit(X_train, y_train)

    return {
        "features": ", ".join(feature_cols),
        "max_depth": max_depth,
        "rmse_train": calculate_rmse(pipe.predict(X_train), y_train),
        "rmse_test": calculate_rmse(pipe.predict(X_test), y_test),
    }

cycles = [
    eval_mce_regressor(mce_filtered, ["place", "platform_released_year"], 3),
    eval_mce_regressor(mce_filtered, ["place", "platform_released_year", "cat_name"], 3),
    eval_mce_regressor(mce_filtered, feature_cols, 6),
]
display(pd.DataFrame(cycles))


### Visualisatie decision tree (opdracht 18)

We plotten de boom van de gefilterde variant (die is meestal het best te interpreteren).


In [ ]:
feature_names = pipe_filt.named_steps["preprocess"].get_feature_names_out()
feature_names = [n.replace("num__", "").replace("cat__", "") for n in feature_names]

plot_tree_regression(
    pipe_filt.named_steps["model"],
    feature_names=feature_names,
)


**Findings (opdracht 18)**  
- RMSE op train is (bijna altijd) lager dan op test: het model ‘kent’ de trainingdata.  
- Als de boom te diep is, zie je vaak **veel lagere train-RMSE** maar **hogere test-RMSE** → overfitting.  
- In deze dataset maken **outliers** RMSE snel groot. Na filtering (95e percentiel) wordt RMSE veel beter interpreteerbaar.  


## Bronnen

- CRISP-DM fase 4 (Modeling): https://medium.com/analytics-vidhya/crisp-dm-phase-4-modeling-phase-b81f2580ff3  
- Penguins dataset (seaborn-data): https://github.com/mwaskom/seaborn-data/blob/master/penguins.csv  
- Scikit-learn Decision Trees (docs): https://scikit-learn.org/stable/modules/tree.html  
